# XiDepth v2.0 — Track 2: XiBlock Lightweight Ablation Training

This notebook trains the **XiDepthNet novel lightweight architecture** (2.36M parameters — 6x smaller than ResNet-18) using identical hyperparameters for rigorous ablation.

### Target Efficiency Metrics:
- Parameter count: 2.36M vs 14.7M (6.2x reduction)
- CPU latency: ~100 ms (~10 FPS)
- Target AbsRel: ~0.132 (within ~10-12% of ResNet-18 baseline)

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# 2. Locate KITTI dataset
import os, glob
input_dirs = glob.glob("/kaggle/input/*kitti*")
dataset_dir = input_dirs[0] if input_dirs else "/kaggle/input/kitti-eigen-split"
print(f"Using dataset directory: {dataset_dir}")


In [ ]:
# 3. Clone or pull XiDepth-Monocular-2.0
!git clone https://github.com/ASMITSARKAR/XiDepth-Monocular-2.0.git xidepth_repo
%cd xidepth_repo
!pip install -r requirements.txt


In [ ]:
# 4. Run Track 2 Training (XiDepthNet lightweight, 20 epochs, AMP FP16)
!python scripts/train.py \
    --model xidepth \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_xidepth \
    --batch_size 12 \
    --epochs 20 \
    --lr 1e-4 \
    --lr_step_size 15 \
    --num_workers 4 \
    --use_amp \
    --log_freq 100


In [ ]:
# 5. Evaluate XiDepthNet on Eigen Benchmark
!python scripts/eval.py \
    --model xidepth \
    --checkpoint /kaggle/working/checkpoints_xidepth/best_model.pth \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/xidepth_results.json
